# Talkie Experiments - Full RunPod Pipeline

This notebook runs all experiments in sequence on RunPod (A100 80GB).

**Estimated time: ~7 hours total**
- Session 1 (Mining + Probing): ~4 hours
- Session 2 (ICL 10 seeds + Qualitative): ~3 hours

Run all cells top to bottom. Each section saves its results independently.

## 0. Setup - Write Updated Files

This section writes all the updated/new files that differ from what's currently on RunPod.

In [1]:
import os
import json
from pathlib import Path

# Set working directory to project root
PROJECT_ROOT = Path(os.getcwd())
print(f"Project root: {PROJECT_ROOT}")
print(f"Files in root: {[f.name for f in PROJECT_ROOT.iterdir() if f.is_file()]}")

Project root: /workspace/Talkie
Files in root: ['run_all_experiments.ipynb', 'qualitative_generations.py', 'probe_analysis.py', 'ocr_ablation.py', 'model_loader.py', 'inspect_checkpoint.py', 'experiment3_probing.py', 'experiment2_icl.py', 'experiment1_syntactic.py', 'diagnose.py', 'debug_nan.py', 'debug_icl2.py', 'debug_icl.py', 'config.py', 'analysis.py']


In [2]:
!pip install tiktoken -q


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [3]:
!pip install tiktoken huggingface_hub datasets scikit-learn numpy pandas matplotlib seaborn scipy tqdm -q


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [6]:
config_content = '''"""Central configuration for all experiments."""

from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────
ROOT = Path(__file__).resolve().parent
RESULTS_DIR = ROOT / "results"
FIGURES_DIR = ROOT / "figures"
DATA_DIR = ROOT / "data"
CACHE_DIR = ROOT / "model_cache"

for d in [RESULTS_DIR, FIGURES_DIR, DATA_DIR, CACHE_DIR]:
    d.mkdir(exist_ok=True)

# ── Model IDs ──────────────────────────────────────────────────────────
VINTAGE_MODEL_ID = "talkie-lm/talkie-1930-13b-base"
MODERN_MODEL_ID = "talkie-lm/talkie-web-13b-base"

MODEL_NAMES = {
    VINTAGE_MODEL_ID: "Talkie-1930",
    MODERN_MODEL_ID: "Talkie-Web",
}

# ── Architecture constants ─────────────────────────────────────────────
N_LAYERS = 40
HIDDEN_DIM = 5120
N_HEADS = 40
HEAD_DIM = 128
VOCAB_SIZE = 65536

# ── Experiment 1: BLiMP ────────────────────────────────────────────────
BLIMP_BATCH_SIZE = 16

BLIMP_PHENOMENA = [
    "anaphor_agreement",
    "argument_structure",
    "binding",
    "control_raising",
    "determiner_noun_agreement",
    "ellipsis",
    "filler_gap",
    "irregular_forms",
    "island_effects",
    "npi_licensing",
    "quantifiers",
    "subject_verb_agreement",
]

# ── Experiment 2: ICL ──────────────────────────────────────────────────
ICL_K_VALUES = [0, 1, 2, 4, 8, 16, 32]
ICL_SEEDS = [42, 123, 456, 789, 101, 202, 303, 404, 505, 606]
ICL_MAX_EVAL_SAMPLES = 500

ICL_TASKS = {
    "sst2": {
        "dataset": "stanfordnlp/sst2",
        "split": "validation",
        "input_key": "sentence",
        "label_key": "label",
        "label_names": ["negative", "positive"],
        "description": "Sentiment analysis (movie reviews)",
    },
    "mnli": {
        "dataset": "nyu-mll/multi_nli",
        "split": "validation_matched",
        "input_keys": ["premise", "hypothesis"],
        "label_key": "label",
        "label_names": ["entailment", "neutral", "contradiction"],
        "description": "Natural language inference",
    },
    "tweet_sentiment": {
        "dataset": "cardiffnlp/tweet_eval",
        "config": "sentiment",
        "split": "test",
        "input_key": "text",
        "label_key": "label",
        "label_names": ["negative", "neutral", "positive"],
        "description": "Tweet sentiment classification",
    },
    "tweet_emotion": {
        "dataset": "cardiffnlp/tweet_eval",
        "config": "emotion",
        "split": "test",
        "input_key": "text",
        "label_key": "label",
        "label_names": ["anger", "joy", "optimism", "sadness"],
        "description": "Tweet emotion classification",
    },
}

# ── Experiment 3: Probing ──────────────────────────────────────────────
PROBE_LAYERS = list(range(0, N_LAYERS + 1, 4))
PROBE_CV_FOLDS = 5
PROBE_MAX_SEQ_LEN = 128

# ── OCR ablation ───────────────────────────────────────────────────────
OCR_ERROR_RATES = [0.01, 0.02, 0.05, 0.10]

# ── Device ─────────────────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    DTYPE = torch.float32  # fp32 required - bf16 breaks logits
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    DTYPE = torch.float16
else:
    DEVICE = torch.device("cpu")
    DTYPE = torch.float32
'''

with open('/workspace/Talkie/config.py', 'w') as f:
    f.write(config_content)
print("Written: config.py")

Written: config.py


In [7]:
"""
Download and load Talkie custom models.

Architecture (from reference src/talkie/model.py):
  - Pre-norm GPT with *parameter-free* F.rms_norm (no learnable γ/β)
  - Per-layer scalar gains (attn_gain, mlp_gain, embed_skip)
  - QK-norm + per-head gain on queries
  - RoPE (base=1e6) positional encoding
  - SwiGLU FFN
  - LM head with weight gain
  - Embedding skip-connection (added after MLP, not with attention)

Checkpoint keys follow `_orig_mod.blocks.{i}.*` naming (torch.compile wrapper).
"""

import base64
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import numpy as np
import tiktoken
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import snapshot_download

import config


# ═══════════════════════════════════════════════════════════════════════
#  Architecture  (matches checkpoint key layout exactly)
# ═══════════════════════════════════════════════════════════════════════

@dataclass
class TalkieConfig:
    n_layer: int = 40
    n_head: int = 40
    n_embd: int = 5120
    head_dim: int = 128
    vocab_size: int = 65536
    max_seq_len: int = 2048
    intermediate_size: int = 13696


class HeadGain(nn.Module):
    def __init__(self, n_head: int):
        super().__init__()
        self.head_g = nn.Parameter(torch.ones(n_head))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * self.head_g.type_as(x).view(1, 1, -1, 1)


class WeightGain(nn.Module):
    def __init__(self):
        super().__init__()
        self.w_g = nn.Parameter(torch.ones(1))

    def forward(self, w: torch.Tensor) -> torch.Tensor:
        return w * self.w_g.type_as(w)


class ActGain(nn.Module):
    def __init__(self, init_value: float):
        super().__init__()
        self.a_g = nn.Parameter(torch.ones(1) * init_value)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * self.a_g.type_as(x)


# ── RoPE (base=1e6, no learnable params) ─────────────────────────────

def _precompute_rotary(seq_len: int, head_dim: int,
                       base: int = 1_000_000) -> tuple[torch.Tensor, torch.Tensor]:
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
    t = torch.arange(seq_len, dtype=torch.float32)
    freqs = torch.outer(t, inv_freq)
    cos = freqs.cos()[None, :, None, :]
    sin = freqs.sin()[None, :, None, :]
    return cos, sin


def apply_rotary_emb(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    d = x.shape[3] // 2
    x1, x2 = x[..., :d], x[..., d:]
    y1 = x1 * cos + x2 * sin
    y2 = x1 * (-sin) + x2 * cos
    return torch.cat([y1, y2], 3).type_as(x)


# ── Attention (with QK-norm and per-head gain on Q) ──────────────────

class Attention(nn.Module):
    def __init__(self, cfg: TalkieConfig):
        super().__init__()
        self.n_head = cfg.n_head
        self.head_dim = cfg.head_dim
        n_state = cfg.n_embd

        self.attn_query = nn.Linear(n_state, n_state, bias=False)
        self.attn_key   = nn.Linear(n_state, n_state, bias=False)
        self.attn_value = nn.Linear(n_state, n_state, bias=False)
        self.attn_resid = nn.Linear(n_state, n_state, bias=False)
        self.head_gain  = HeadGain(cfg.n_head)

    def forward(self, x: torch.Tensor, cos_sin: tuple) -> torch.Tensor:
        B, T, _ = x.shape
        q = self.attn_query(x).view(B, T, self.n_head, self.head_dim)
        k = self.attn_key(x).view(B, T, self.n_head, self.head_dim)
        v = self.attn_value(x).view(B, T, self.n_head, self.head_dim)

        cos, sin = cos_sin
        q = apply_rotary_emb(q, cos, sin)
        k = apply_rotary_emb(k, cos, sin)

        q = F.rms_norm(q, (q.size(-1),))
        k = F.rms_norm(k, (k.size(-1),))
        q = self.head_gain(q)

        y = F.scaled_dot_product_attention(
            q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2), is_causal=True
        )
        y = y.transpose(1, 2).contiguous().view(B, T, -1)
        return self.attn_resid(y)


# ── SwiGLU FFN ────────────────────────────────────────────────────────

class MLP(nn.Module):
    def __init__(self, cfg: TalkieConfig):
        super().__init__()
        self.mlp_gate   = nn.Linear(cfg.n_embd, cfg.intermediate_size, bias=False)
        self.mlp_linear = nn.Linear(cfg.n_embd, cfg.intermediate_size, bias=False)
        self.mlp_resid  = nn.Linear(cfg.intermediate_size, cfg.n_embd, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.mlp_resid(F.silu(self.mlp_gate(x)) * self.mlp_linear(x))


# ── Transformer block (pre-norm with parameter-free RMS norm) ────────

class TransformerBlock(nn.Module):
    def __init__(self, cfg: TalkieConfig):
        super().__init__()
        self.attn       = Attention(cfg)
        self.mlp        = MLP(cfg)
        self.attn_gain  = ActGain((2 * cfg.n_layer) ** -0.5)
        self.mlp_gain   = ActGain((2 * cfg.n_layer) ** -0.5)
        self.embed_skip = ActGain(0.0)

    def forward(self, e_x: torch.Tensor, x: torch.Tensor,
                cos_sin: tuple) -> torch.Tensor:
        x = x + self.attn_gain(self.attn(F.rms_norm(x, (x.shape[-1],)), cos_sin))
        x = x + self.mlp_gain(self.mlp(F.rms_norm(x, (x.shape[-1],))))
        x = x + self.embed_skip(e_x)
        return x


# ── Full model ────────────────────────────────────────────────────────

class TalkieModel(nn.Module):
    def __init__(self, cfg: TalkieConfig):
        super().__init__()
        self.cfg = cfg
        self.embed = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layer)])
        self.lm_head = nn.Parameter(torch.zeros(cfg.vocab_size, cfg.n_embd))
        self.lm_head_gain = WeightGain()

        cos, sin = _precompute_rotary(cfg.max_seq_len, cfg.head_dim)
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)

    def forward(self, input_ids: torch.Tensor, output_hidden_states: bool = False,
                last_positions: Optional[torch.Tensor] = None):
        B, T = input_ids.shape
        cos_sin = self.cos[:, :T].to(input_ids.device), self.sin[:, :T].to(input_ids.device)

        x = self.embed(input_ids)
        x = F.rms_norm(x, (x.shape[-1],))
        e_x = x

        hidden_states = [x] if output_hidden_states else None

        for block in self.blocks:
            x = block(e_x, x, cos_sin)
            if output_hidden_states:
                hidden_states.append(x)

        x = F.rms_norm(x, (x.shape[-1],))

        if last_positions is not None:
            x_sel = x[torch.arange(B, device=x.device), last_positions]
            logits = F.linear(x_sel, self.lm_head_gain(self.lm_head)).float()
        else:
            logits = F.linear(x, self.lm_head_gain(self.lm_head)).float()

        if output_hidden_states:
            hidden_states.append(x)
            return logits, tuple(hidden_states)
        return (logits,)


# ═══════════════════════════════════════════════════════════════════════
#  Checkpoint loading
# ═══════════════════════════════════════════════════════════════════════

def _strip_prefix(state_dict: dict, prefix: str = "_orig_mod.") -> dict:
    """Remove torch.compile wrapper prefix from all keys."""
    return {
        (k[len(prefix):] if k.startswith(prefix) else k): v
        for k, v in state_dict.items()
    }


def _detect_config(state_dict: dict) -> TalkieConfig:
    cfg = TalkieConfig()
    for k, v in state_dict.items():
        if not hasattr(v, "shape"):
            continue
        if k == "embed.weight":
            cfg.vocab_size, cfg.n_embd = v.shape
        elif "mlp_gate.weight" in k:
            cfg.intermediate_size = v.shape[0]
        elif "head_gain.head_g" in k:
            cfg.n_head = v.shape[0]

    layer_ids = set()
    for k in state_dict:
        if k.startswith("blocks."):
            idx = k.split(".")[1]
            if idx.isdigit():
                layer_ids.add(int(idx))
    if layer_ids:
        cfg.n_layer = max(layer_ids) + 1
    cfg.head_dim = cfg.n_embd // cfg.n_head

    print(f"[detect] n_layer={cfg.n_layer}  n_embd={cfg.n_embd}  n_head={cfg.n_head}  "
          f"head_dim={cfg.head_dim}  intermediate={cfg.intermediate_size}  vocab={cfg.vocab_size}")
    return cfg


def inspect_checkpoint(ckpt_path):
    raw = torch.load(str(ckpt_path), map_location="cpu", weights_only=False)
    if isinstance(raw, dict):
        for w in ["model", "state_dict"]:
            if w in raw and isinstance(raw[w], dict):
                raw = raw[w]
                print(f"[inspect] Unwrapped '{w}'")
                break
    print(f"\n{'Key':<70} {'Shape':<25} Dtype")
    print("-" * 110)
    for k, v in sorted(raw.items()):
        s = tuple(v.shape) if hasattr(v, "shape") else "scalar"
        d = v.dtype if hasattr(v, "dtype") else type(v).__name__
        print(f"{k:<70} {str(s):<25} {d}")
    print(f"\nTotal keys: {len(raw)}")


# ═══════════════════════════════════════════════════════════════════════
#  Tokeniser
# ═══════════════════════════════════════════════════════════════════════

def load_tiktoken_tokeniser(repo_dir: Path, expected_vocab: int = 65536) -> tiktoken.Encoding:
    """Load tiktoken tokeniser from vocab.txt (base64+rank format)."""
    vocab_path = repo_dir / "vocab.txt"
    if not vocab_path.exists():
        print("[tokeniser] WARNING: No vocab.txt, using cl100k_base fallback.")
        return tiktoken.get_encoding("cl100k_base")

    # Parse base64+rank format: each line is "BASE64_TOKEN RANK"
    mergeable_ranks = {}
    with open(vocab_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) >= 2:
                try:
                    token_bytes = base64.b64decode(parts[0])
                    rank = int(parts[1])
                    if rank < expected_vocab:
                        mergeable_ranks[token_bytes] = rank
                except Exception:
                    continue

    print(f"[tokeniser] Loaded {len(mergeable_ranks)} tokens "
          f"(max rank {max(mergeable_ranks.values()) if mergeable_ranks else 0})")
    return _build_encoding(repo_dir.name, mergeable_ranks)


def _build_encoding(name: str, mergeable_ranks: dict) -> tiktoken.Encoding:
    return tiktoken.Encoding(
        name=f"talkie-{name}",
        pat_str=r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]+[\r\n]*|\s*[\r\n]+|\s+(?!\S)|\s+""",
        mergeable_ranks=mergeable_ranks,
        special_tokens={"<|endoftext|>": len(mergeable_ranks)},
    )


# ═══════════════════════════════════════════════════════════════════════
#  Download
# ═══════════════════════════════════════════════════════════════════════

def download_model_repo(model_id: str) -> Path:
    local_dir = config.CACHE_DIR / model_id.replace("/", "--")
    if local_dir.exists() and any(local_dir.iterdir()):
        print(f"[model_loader] Using cached repo: {local_dir}")
        return local_dir
    print(f"[model_loader] Downloading {model_id} ...")
    snapshot_download(repo_id=model_id, local_dir=str(local_dir))
    return local_dir


def _find_checkpoint(repo_dir: Path) -> Path:
    for pattern in ["*.ckpt", "*.pt", "*.pth", "*.bin"]:
        matches = sorted(repo_dir.glob(pattern), key=lambda p: p.stat().st_size, reverse=True)
        if matches:
            return matches[0]
    raise FileNotFoundError(f"No checkpoint found in {repo_dir}")


# ═══════════════════════════════════════════════════════════════════════
#  ModelWrapper
# ═══════════════════════════════════════════════════════════════════════

class ModelWrapper:
    def __init__(self, model: TalkieModel, tokeniser: tiktoken.Encoding, model_id: str):
        self.model = model
        self.tokeniser = tokeniser
        self.model_id = model_id
        self.name = config.MODEL_NAMES.get(model_id, model_id)
        self.device = config.DEVICE
        self.dtype = config.DTYPE

    @torch.no_grad()
    def log_likelihood(self, text: str) -> float:
        token_ids = self.tokeniser.encode(text)
        if not token_ids:
            return 0.0
        input_ids = torch.tensor([token_ids], device=self.device)
        logits = self.model(input_ids)[0]
        shift_logits = logits[:, :-1, :]
        shift_labels = input_ids[:, 1:]
        log_probs = F.log_softmax(shift_logits, dim=-1)
        return log_probs.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1).sum().item()

    @torch.no_grad()
    def log_likelihood_per_token(self, text: str) -> tuple[float, int]:
        token_ids = self.tokeniser.encode(text)
        n = len(token_ids)
        if n <= 1:
            return 0.0, max(n, 1)
        input_ids = torch.tensor([token_ids], device=self.device)
        logits = self.model(input_ids)[0]
        shift_logits = logits[:, :-1, :]
        shift_labels = input_ids[:, 1:]
        log_probs = F.log_softmax(shift_logits, dim=-1)
        return log_probs.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1).sum().item(), n

    def score_pair(self, good: str, bad: str) -> dict:
        ll_good, n_good = self.log_likelihood_per_token(good)
        ll_bad, n_bad = self.log_likelihood_per_token(bad)
        return {
            "ll_good": ll_good, "ll_bad": ll_bad,
            "n_tokens_good": n_good, "n_tokens_bad": n_bad,
            "ll_good_norm": ll_good / n_good, "ll_bad_norm": ll_bad / n_bad,
            "correct": ll_good > ll_bad,
            "correct_norm": (ll_good / n_good) > (ll_bad / n_bad),
        }

    @torch.no_grad()
    def batch_log_likelihood(self, texts: list[str],
                             batch_size: int = 256) -> list[tuple[float, int]]:
        all_token_ids = [self.tokeniser.encode(t) for t in texts]
        results = [None] * len(texts)

        for start in range(0, len(texts), batch_size):
            batch_ids = all_token_ids[start:start + batch_size]
            lengths = [len(ids) for ids in batch_ids]
            max_len = max(lengths)

            padded = torch.zeros(len(batch_ids), max_len, dtype=torch.long,
                                 device=self.device)
            for j, ids in enumerate(batch_ids):
                padded[j, :len(ids)] = torch.tensor(ids, dtype=torch.long)

            logits = self.model(padded)[0]
            log_probs = F.log_softmax(logits[:, :-1, :], dim=-1)
            targets = padded[:, 1:].unsqueeze(-1)
            token_lps = log_probs.gather(2, targets).squeeze(-1)

            for j, n in enumerate(lengths):
                if n <= 1:
                    results[start + j] = (0.0, max(n, 1))
                else:
                    results[start + j] = (token_lps[j, :n - 1].sum().item(), n)

        return results

    @torch.no_grad()
    def batch_conditional_ll(self, prompts: list[str], completions: list[str],
                             batch_size: int = 32) -> list[tuple[float, int]]:
        """Log P(completion | prompt) for each (prompt, completion) pair.

        Returns list of (conditional_ll, n_completion_tokens).
        """
        prompt_ids = [self.tokeniser.encode(p) for p in prompts]
        full_ids = [self.tokeniser.encode(p + c) for p, c in zip(prompts, completions)]
        results = [None] * len(prompts)

        for start in range(0, len(prompts), batch_size):
            end = min(start + batch_size, len(prompts))
            batch_full = full_ids[start:end]
            batch_prompt_lens = [len(prompt_ids[start + j]) for j in range(end - start)]
            lengths = [len(ids) for ids in batch_full]
            max_len = max(lengths)
            B = len(batch_full)

            padded = torch.zeros(B, max_len, dtype=torch.long, device=self.device)
            for j, ids in enumerate(batch_full):
                padded[j, :len(ids)] = torch.tensor(ids, dtype=torch.long)

            logits = self.model(padded)[0]
            log_probs = F.log_softmax(logits[:, :-1, :], dim=-1)
            targets = padded[:, 1:].unsqueeze(-1)
            token_lps = log_probs.gather(2, targets).squeeze(-1)

            for j in range(B):
                prompt_len = batch_prompt_lens[j]
                seq_len = lengths[j]
                n_completion = seq_len - prompt_len
                if n_completion <= 0:
                    results[start + j] = (0.0, 0)
                else:
                    cond_ll = token_lps[j, prompt_len - 1:seq_len - 1].sum().item()
                    results[start + j] = (cond_ll, n_completion)

        return results

    @torch.no_grad()
    def generate(self, prompt: str, max_new_tokens: int = 64,
                 temperature: float = 0.0, stop_tokens: Optional[list[str]] = None) -> str:
        token_ids = self.tokeniser.encode(prompt)
        generated = list(token_ids)
        eos = self.tokeniser.encode("<|endoftext|>", allowed_special={"<|endoftext|>"})
        eos_id = eos[0] if eos else None

        for _ in range(max_new_tokens):
            input_ids = torch.tensor([generated], device=self.device)
            logits = self.model(input_ids)[0]
            next_logits = logits[:, -1, :]
            if temperature <= 0:
                tok = next_logits.argmax(-1).item()
            else:
                tok = torch.multinomial(F.softmax(next_logits / temperature, -1), 1).item()
            if tok == eos_id:
                break
            generated.append(tok)
            if stop_tokens:
                decoded = self.tokeniser.decode(generated[len(token_ids):])
                if any(s in decoded for s in stop_tokens):
                    break
        return self.tokeniser.decode(generated[len(token_ids):])

    @torch.no_grad()
    def extract_hidden_states(self, text: str,
                              layer_indices: Optional[list[int]] = None) -> dict[int, torch.Tensor]:
        if layer_indices is None:
            layer_indices = config.PROBE_LAYERS
        token_ids = self.tokeniser.encode(text)[:config.PROBE_MAX_SEQ_LEN]
        input_ids = torch.tensor([token_ids], device=self.device)
        logits, hidden_states = self.model(input_ids, output_hidden_states=True)
        result = {}
        for idx in layer_indices:
            if idx < len(hidden_states):
                result[idx] = hidden_states[idx].squeeze(0).mean(dim=0).cpu().float()
        return result

    @torch.no_grad()
    def get_next_token_probs(self, prompt: str, target_tokens: list[str]) -> dict[str, float]:
        token_ids = self.tokeniser.encode(prompt)
        input_ids = torch.tensor([token_ids], device=self.device)
        logits = self.model(input_ids)[0]
        probs = F.softmax(logits[:, -1, :], dim=-1).squeeze(0)
        result = {}
        for tok_str in target_tokens:
            ids = self.tokeniser.encode(tok_str)
            result[tok_str] = probs[ids[0]].item() if ids else 0.0
        return result

    @torch.no_grad()
    def batch_next_token_probs(self, prompts: list[str],
                               target_token_ids: list[int],
                               batch_size: int = 32) -> torch.Tensor:
        """Batched next-token probabilities. Returns [n_prompts, n_targets]."""
        all_ids = [self.tokeniser.encode(p) for p in prompts]
        target_idx = torch.tensor(target_token_ids, device=self.device)
        out = torch.zeros(len(prompts), len(target_token_ids))

        for start in range(0, len(prompts), batch_size):
            batch_ids = all_ids[start:start + batch_size]
            lengths = [len(ids) for ids in batch_ids]
            max_len = max(lengths)
            B = len(batch_ids)

            padded = torch.zeros(B, max_len, dtype=torch.long, device=self.device)
            for j, ids in enumerate(batch_ids):
                padded[j, :len(ids)] = torch.tensor(ids, dtype=torch.long)

            last_pos = torch.tensor([l - 1 for l in lengths], device=self.device)
            logits = self.model(padded, last_positions=last_pos)[0]
            probs = F.softmax(logits, dim=-1)
            out[start:start + B] = probs[:, target_idx].cpu()

        return out


# ═══════════════════════════════════════════════════════════════════════
#  Main entry point
# ═══════════════════════════════════════════════════════════════════════

def load_model(model_id: str) -> ModelWrapper:
    repo_dir = download_model_repo(model_id)
    ckpt_path = _find_checkpoint(repo_dir)
    print(f"[model_loader] Checkpoint: {ckpt_path.name} ({ckpt_path.stat().st_size / 1e9:.1f} GB)")

    # Load & unwrap checkpoint
    print("[model_loader] Loading checkpoint into RAM ...")
    raw = torch.load(str(ckpt_path), map_location="cpu", weights_only=False)
    state_dict = raw
    for wrapper in ["model", "state_dict", "model_state_dict"]:
        if isinstance(state_dict, dict) and wrapper in state_dict and isinstance(state_dict[wrapper], dict):
            state_dict = state_dict[wrapper]
            break

    # Strip _orig_mod. prefix from torch.compile
    state_dict = _strip_prefix(state_dict, "_orig_mod.")

    # Detect architecture and build model
    arch_cfg = _detect_config(state_dict)
    model = TalkieModel(arch_cfg)

    # Load weights
    model_sd = model.state_dict()
    ckpt_keys = set(state_dict.keys())
    model_keys = set(model_sd.keys())

    matched = ckpt_keys & model_keys
    missing_in_ckpt = model_keys - ckpt_keys
    extra_in_ckpt = ckpt_keys - model_keys

    for k in matched:
        if model_sd[k].shape == state_dict[k].shape:
            model_sd[k] = state_dict[k]
        else:
            print(f"  Shape mismatch: {k}  model={model_sd[k].shape}  ckpt={state_dict[k].shape}")

    model.load_state_dict(model_sd, strict=False)
    del raw, state_dict

    print(f"[model_loader] Loaded {len(matched)}/{len(model_keys)} params")
    if missing_in_ckpt:
        # Filter out non-persistent buffers (rotary caches)
        real_missing = {k for k in missing_in_ckpt if "rotary" not in k and "cached" not in k}
        if real_missing:
            print(f"  Missing in ckpt: {sorted(real_missing)[:5]}")
    if extra_in_ckpt:
        print(f"  Extra in ckpt:   {sorted(extra_in_ckpt)[:5]}")

    model = model.to(dtype=torch.float32).to(config.DEVICE)
    model.eval()
    n_params = sum(p.numel() for p in model.parameters())
    print(f"[model_loader] {n_params/1e9:.2f}B params on {config.DEVICE} (float32)")

    # Tokeniser
    tokeniser = load_tiktoken_tokeniser(repo_dir, expected_vocab=arch_cfg.vocab_size)
    print(f"[model_loader] Tokeniser: vocab={tokeniser.n_vocab}")

    return ModelWrapper(model, tokeniser, model_id)

In [8]:
# Patch model_loader.py on disk
with open("/workspace/Talkie/model_loader.py", "r") as f:
    content = f.read()

content = content.replace('torch.bfloat16', 'torch.float32')
content = content.replace('(bfloat16)', '(float32)')

with open("/workspace/Talkie/model_loader.py", "w") as f:
    f.write(content)

print("Patched!")

# Reload both modules
import importlib
import config
importlib.reload(config)
import model_loader
importlib.reload(model_loader)
from model_loader import load_model

print(f"config.DTYPE = {config.DTYPE}")

Patched!
config.DTYPE = torch.float32


In [9]:
# Write updated data/temporal_facts.json (expanded to 200+ per category)
temporal_facts = json.loads(open(PROJECT_ROOT / 'data' / 'temporal_facts.json').read()) if (PROJECT_ROOT / 'data' / 'temporal_facts.json').exists() else None

# Check if it's already expanded
if temporal_facts and len(temporal_facts.get('pre_1930_true', [])) >= 200:
    print(f"temporal_facts.json already expanded: {len(temporal_facts['pre_1930_true'])} pre_1930_true items")
else:
    print("Writing expanded temporal_facts.json...")
    # The expanded dataset is embedded below
    temporal_facts_expanded = TEMPORAL_FACTS_DATA  # Will be defined in next cell
    (PROJECT_ROOT / 'data').mkdir(exist_ok=True)
    with open(PROJECT_ROOT / 'data' / 'temporal_facts.json', 'w') as f:
        json.dump(temporal_facts_expanded, f, indent=2, ensure_ascii=False)
    print("Written: data/temporal_facts.json")

temporal_facts.json already expanded: 200 pre_1930_true items


In [10]:
# Write the expanded temporal_facts.json directly
import urllib.request

# Since the file is large, write it cell by cell
(PROJECT_ROOT / 'data').mkdir(exist_ok=True)

# Check current size
tf_path = PROJECT_ROOT / 'data' / 'temporal_facts.json'
if tf_path.exists():
    with open(tf_path) as f:
        current = json.load(f)
    sizes = {k: len(v) for k, v in current.items()}
    print(f"Current temporal_facts.json sizes: {sizes}")
    if sizes.get('pre_1930_true', 0) >= 200:
        print("Already expanded! Skipping.")
    else:
        print("Need to write expanded version - see next cell")
else:
    print("File not found - will write in next cell")

Current temporal_facts.json sizes: {'pre_1930_true': 200, 'pre_1930_false': 199, 'post_1930_true': 199, 'post_1930_false': 199, 'leaked_knowledge': 15}
Already expanded! Skipping.


In [12]:
# Restore dataset from write_dataset.py (handles null->None conversion)
import subprocess
result = subprocess.run(['python', 'write_dataset.py'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('ERROR:', result.stderr)

In [13]:
# Verify the temporal_facts.json is correct
with open(PROJECT_ROOT / 'data' / 'temporal_facts.json') as f:
    tf = json.load(f)
print("Temporal facts dataset sizes:")
for k, v in tf.items():
    print(f"  {k}: {len(v)}")
assert len(tf['pre_1930_true']) >= 200, f"Expected 200+ pre_1930_true, got {len(tf['pre_1930_true'])}"
print("\nDataset verification PASSED")

Temporal facts dataset sizes:
  pre_1930_true: 200
  pre_1930_false: 199
  post_1930_true: 199
  post_1930_false: 199
  leaked_knowledge: 15

Dataset verification PASSED


## 1. Mine Leaked Knowledge (~30 min)

Tests ~80 borderline post-1930 facts on Talkie-1930. Facts correctly completed are leaked knowledge candidates.

In [16]:
import torch
import importlib
import config
importlib.reload(config)
from model_loader import load_model

print(f"Device: {config.DEVICE}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
CUDA available: True
GPU: NVIDIA A100 80GB PCIe
Memory: 85.1 GB


In [17]:
!nvidia-smi

Sun May 17 01:10:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.20             Driver Version: 580.126.20     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          On  |   00000000:41:00.0 Off |                    0 |
| N/A   37C    P0             46W /  300W |       4MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [18]:
# Post-1930 facts to test, focusing on 1930-1945 borderline period
BORDERLINE_FACTS = [
    # 1930-1935 (most likely to leak through borderline dates)
    {"prompt": "The planet Pluto was discovered in the year", "expected": "1930", "year": 1930},
    {"prompt": "The Empire State Building was completed in", "expected": "1931", "year": 1931},
    {"prompt": "Franklin Roosevelt was first elected president in", "expected": "1932", "year": 1932},
    {"prompt": "Adolf Hitler became Chancellor of Germany in", "expected": "1933", "year": 1933},
    {"prompt": "The Dust Bowl devastated the American", "expected": "Great Plains", "year": 1934},
    {"prompt": "The Hoover Dam was completed in the year", "expected": "1935", "year": 1935},
    {"prompt": "Jesse Owens won four gold medals at the Olympics in", "expected": "Berlin", "year": 1936},
    {"prompt": "The Hindenburg disaster occurred in", "expected": "1937", "year": 1937},
    {"prompt": "Nylon was first commercially produced by", "expected": "DuPont", "year": 1938},
    {"prompt": "Germany invaded Poland in September", "expected": "1939", "year": 1939},
    {"prompt": "The Golden Gate Bridge is located in", "expected": "San Francisco", "year": 1937},
    {"prompt": "The New Deal was a series of programs by President", "expected": "Roosevelt", "year": 1933},
    {"prompt": "Amelia Earhart disappeared over the Pacific in", "expected": "1937", "year": 1937},
    {"prompt": "The Spanish Civil War began in", "expected": "1936", "year": 1936},
    {"prompt": "Penicillin was first used as a medicine by Alexander", "expected": "Fleming", "year": 1929},
    {"prompt": "The stock market crashed in October", "expected": "1929", "year": 1929},
    {"prompt": "The Great Depression began in the year", "expected": "1929", "year": 1929},
    {"prompt": "Mahatma Gandhi led the Salt March in", "expected": "1930", "year": 1930},
    {"prompt": "The Nazis held the Nuremberg rallies in the city of", "expected": "Nuremberg", "year": 1933},
    {"prompt": "Japan invaded Manchuria in", "expected": "1931", "year": 1931},

    # 1935-1945
    {"prompt": "World War II began in the year", "expected": "1939", "year": 1939},
    {"prompt": "The Battle of Britain was fought in", "expected": "1940", "year": 1940},
    {"prompt": "Japan attacked Pearl Harbor on December 7,", "expected": "1941", "year": 1941},
    {"prompt": "The Battle of Stalingrad took place in", "expected": "1942", "year": 1942},
    {"prompt": "D-Day, the Allied invasion of Normandy, occurred in", "expected": "1944", "year": 1944},
    {"prompt": "The United Nations was founded in", "expected": "1945", "year": 1945},
    {"prompt": "The atomic bomb was first tested at", "expected": "Trinity", "year": 1945},
    {"prompt": "Anne Frank wrote her diary while hiding in", "expected": "Amsterdam", "year": 1942},
    {"prompt": "The Manhattan Project developed the first", "expected": "atomic", "year": 1942},
    {"prompt": "Radar technology was crucial in the Battle of", "expected": "Britain", "year": 1940},

    # 1945-1960 (less likely to leak)
    {"prompt": "The Marshall Plan helped rebuild", "expected": "Europe", "year": 1948},
    {"prompt": "The State of Israel was established in", "expected": "1948", "year": 1948},
    {"prompt": "NATO was established in the year", "expected": "1949", "year": 1949},
    {"prompt": "The Korean War began in", "expected": "1950", "year": 1950},
    {"prompt": "Queen Elizabeth II became queen in", "expected": "1952", "year": 1952},
    {"prompt": "The double helix structure of DNA was discovered in", "expected": "1953", "year": 1953},
    {"prompt": "Rosa Parks refused to give up her seat in", "expected": "1955", "year": 1955},
    {"prompt": "Sputnik, the first artificial satellite, was launched by the", "expected": "Soviet", "year": 1957},
    {"prompt": "The European Economic Community was established in", "expected": "1957", "year": 1957},
    {"prompt": "Fidel Castro came to power in Cuba in", "expected": "1959", "year": 1959},

    # 1960-1990 (negative controls)
    {"prompt": "The Cuban Missile Crisis occurred in", "expected": "1962", "year": 1962},
    {"prompt": "John F. Kennedy was assassinated in", "expected": "1963", "year": 1963},
    {"prompt": "The Civil Rights Act was signed in", "expected": "1964", "year": 1964},
    {"prompt": "The first heart transplant was performed by", "expected": "Barnard", "year": 1967},
    {"prompt": "Woodstock music festival took place in", "expected": "1969", "year": 1969},
    {"prompt": "The Watergate scandal involved President", "expected": "Nixon", "year": 1972},
    {"prompt": "The Vietnam War ended in", "expected": "1975", "year": 1975},
    {"prompt": "The Camp David Accords were signed by", "expected": "Carter", "year": 1978},
    {"prompt": "Margaret Thatcher became Prime Minister in", "expected": "1979", "year": 1979},
    {"prompt": "The AIDS epidemic was first identified in", "expected": "1981", "year": 1981},
    {"prompt": "The Falklands War was between Britain and", "expected": "Argentina", "year": 1982},
    {"prompt": "The Internet was originally developed by", "expected": "DARPA", "year": 1969},
    {"prompt": "The first personal computer was the", "expected": "Apple", "year": 1977},
    {"prompt": "Mikhail Gorbachev introduced the policy of", "expected": "perestroika", "year": 1986},
    {"prompt": "Nelson Mandela was released from prison in", "expected": "1990", "year": 1990},

    # Additional borderline 1928-1932
    {"prompt": "Alexander Fleming discovered penicillin in", "expected": "1928", "year": 1928},
    {"prompt": "The first Academy Awards ceremony was held in", "expected": "1929", "year": 1929},
    {"prompt": "The Chrysler Building was completed in New York in", "expected": "1930", "year": 1930},
    {"prompt": "The Star-Spangled Banner became the national anthem in", "expected": "1931", "year": 1931},
    {"prompt": "Aldous Huxley published Brave New World in", "expected": "1932", "year": 1932},
    {"prompt": "The first FIFA World Cup was held in", "expected": "Uruguay", "year": 1930},
    {"prompt": "The Smoot-Hawley Tariff Act was signed in", "expected": "1930", "year": 1930},

    # Science near boundary
    {"prompt": "Edwin Hubble showed that the universe is", "expected": "expanding", "year": 1929},
    {"prompt": "The neutron was discovered by James", "expected": "Chadwick", "year": 1932},
    {"prompt": "Dirac predicted the existence of the", "expected": "positron", "year": 1931},
    {"prompt": "Kurt Godel published his incompleteness theorems in", "expected": "1931", "year": 1931},
    {"prompt": "Heavy water was discovered in", "expected": "1932", "year": 1932},

    # Culture near boundary
    {"prompt": "The first talking motion picture was The Jazz", "expected": "Singer", "year": 1927},
    {"prompt": "Mickey Mouse first appeared in the cartoon Steamboat", "expected": "Willie", "year": 1928},
    {"prompt": "Gone with the Wind was published by Margaret", "expected": "Mitchell", "year": 1936},
    {"prompt": "Walt Disney released the first full-length animated film Snow White in", "expected": "1937", "year": 1937},
]

print(f"Total borderline facts to test: {len(BORDERLINE_FACTS)}")

Total borderline facts to test: 71


In [19]:
# Run the mining
print("=" * 60)
print("MINING FOR LEAKED KNOWLEDGE IN TALKIE-1930")
print("=" * 60)

model = load_model(config.VINTAGE_MODEL_ID)

results = []
correct_by_decade = {}

for item in BORDERLINE_FACTS:
    completion = model.generate(
        item["prompt"], max_new_tokens=30, temperature=0.0
    )
    generated = completion.strip()
    hit = item["expected"].lower() in generated.lower()
    mark = "Y" if hit else "N"

    decade = (item["year"] // 10) * 10
    if decade not in correct_by_decade:
        correct_by_decade[decade] = {"correct": 0, "total": 0}
    correct_by_decade[decade]["total"] += 1
    if hit:
        correct_by_decade[decade]["correct"] += 1

    result = {
        "prompt": item["prompt"],
        "expected": item["expected"],
        "year": item["year"],
        "completion": generated[:100],
        "correct": hit,
    }
    results.append(result)
    print(f"  [{mark}] ({item['year']}) \"{item['prompt']}\"")
    print(f"       -> \"{generated[:70]}\"")

del model
torch.cuda.empty_cache()

# Summary
print(f"\n{'='*60}")
print("SUMMARY BY DECADE")
print(f"{'='*60}")
for decade in sorted(correct_by_decade.keys()):
    d = correct_by_decade[decade]
    pct = d["correct"] / d["total"] * 100 if d["total"] > 0 else 0
    print(f"  {decade}s: {d['correct']}/{d['total']} ({pct:.0f}%)")

leaked_candidates = [r for r in results if r["correct"]]
print(f"\n  Total leaked candidates: {len(leaked_candidates)}")
print(f"  Total tested: {len(results)}")

# Save
config.RESULTS_DIR.mkdir(exist_ok=True)
out_path = config.RESULTS_DIR / "leaked_candidates.json"
with open(out_path, "w") as f:
    json.dump({
        "all_results": results,
        "leaked_candidates": leaked_candidates,
        "by_decade": {str(k): v for k, v in correct_by_decade.items()},
    }, f, indent=2, ensure_ascii=False)
print(f"\n  Saved to {out_path}")

MINING FOR LEAKED KNOWLEDGE IN TALKIE-1930
[model_loader] Using cached repo: /workspace/Talkie/model_cache/talkie-lm--talkie-1930-13b-base
[model_loader] Checkpoint: final.ckpt (53.1 GB)
[model_loader] Loading checkpoint into RAM ...
[detect] n_layer=40  n_embd=5120  n_head=40  head_dim=128  intermediate=13696  vocab=65536
[model_loader] Loaded 443/443 params
[model_loader] 13.28B params on cuda (float32)
[tokeniser] Loaded 65536 tokens (max rank 65535)
[model_loader] Tokeniser: vocab=65537
  [N] (1930) "The planet Pluto was discovered in the year"
       -> "1892 by M. Charlois, of Nice, France. It is a very small planet, being"
  [N] (1931) "The Empire State Building was completed in"
       -> "1892, and is one of the most imposing structures in the city. It is bu"
  [N] (1932) "Franklin Roosevelt was first elected president in"
       -> "1924. He was reelected in 1928. He was born in New York City in 1879. "
  [Y] (1933) "Adolf Hitler became Chancellor of Germany in"
       -> "19

## 2. Expand Dataset with Leaked Candidates (~1 min)

In [20]:
# Load current dataset and add leaked candidates
data_path = config.DATA_DIR / "temporal_facts.json"
with open(data_path) as f:
    data = json.load(f)

print(f"Current dataset sizes:")
for key, items in data.items():
    print(f"  {key}: {len(items)}")

# Add leaked candidates
existing_texts = {item["text"] for item in data["leaked_knowledge"]}
added = 0
for c in leaked_candidates:
    text = f"{c['prompt']} {c['expected']}"
    if text not in existing_texts:
        data["leaked_knowledge"].append({
            "text": text,
            "year": c["year"],
            "source": "mined",
            "domain": "general",
        })
        existing_texts.add(text)
        added += 1

print(f"\n  Added {added} new leaked items (total: {len(data['leaked_knowledge'])})")

# Save updated dataset
with open(data_path, "w") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)
print(f"  Updated dataset saved to {data_path}")

print(f"\nFinal dataset sizes:")
for key, items in data.items():
    print(f"  {key}: {len(items)}")

Current dataset sizes:
  pre_1930_true: 200
  pre_1930_false: 199
  post_1930_true: 199
  post_1930_false: 199
  leaked_knowledge: 15

  Added 7 new leaked items (total: 22)
  Updated dataset saved to /workspace/Talkie/data/temporal_facts.json

Final dataset sizes:
  pre_1930_true: 200
  pre_1930_false: 199
  post_1930_true: 199
  post_1930_false: 199
  leaked_knowledge: 22


## 3. Experiment 3: Probing with Pipeline Fix (~2.5 hours)

Re-runs probing with:
- StandardScaler inside Pipeline (no train/test leakage)
- Expanded dataset (200+ items per category)
- Updated leaked class (30-50 items from mining)

In [21]:
# Reload config and modules to pick up changes
import importlib
import config
importlib.reload(config)

from experiment3_probing import run_experiment3

print("Running Experiment 3 (Probing) with Pipeline fix + expanded data...")
print(f"Probe layers: {config.PROBE_LAYERS}")
print(f"CV folds: {config.PROBE_CV_FOLDS}")

probing_results = run_experiment3()
print("\nExperiment 3 COMPLETE")

Running Experiment 3 (Probing) with Pipeline fix + expanded data...
Probe layers: [0, 4, 8, 12, 16, 20, 24, 28, 32, 36, 40]
CV folds: 5

Probing: Talkie-1930
[model_loader] Using cached repo: /workspace/Talkie/model_cache/talkie-lm--talkie-1930-13b-base
[model_loader] Checkpoint: final.ckpt (53.1 GB)
[model_loader] Loading checkpoint into RAM ...
[detect] n_layer=40  n_embd=5120  n_head=40  head_dim=128  intermediate=13696  vocab=65536
[model_loader] Loaded 443/443 params
[model_loader] 13.28B params on cuda (float32)
[tokeniser] Loaded 65536 tokens (max rank 65535)
[model_loader] Tokeniser: vocab=65537

  probe_a_veracity (n=797, classes=2)
    Extracting hidden states...


    Extracting hidden states: 100%|██████████| 797/797 [00:57<00:00, 13.87it/s]


    Running probes...
      Layer  0: 0.645 ± 0.028
      Layer  4: 0.671 ± 0.032
      Layer  8: 0.689 ± 0.032
      Layer 12: 0.665 ± 0.032
      Layer 16: 0.688 ± 0.021
      Layer 20: 0.666 ± 0.029
      Layer 24: 0.701 ± 0.038
      Layer 28: 0.666 ± 0.033
      Layer 32: 0.670 ± 0.021
      Layer 36: 0.635 ± 0.026
      Layer 40: 0.629 ± 0.019

  probe_b_temporal (n=399, classes=2)
    Extracting hidden states...


    Extracting hidden states: 100%|██████████| 399/399 [00:28<00:00, 13.79it/s]


    Running probes...
      Layer  0: 0.945 ± 0.028
      Layer  4: 0.945 ± 0.032
      Layer  8: 0.942 ± 0.025
      Layer 12: 0.955 ± 0.022
      Layer 16: 0.982 ± 0.013
      Layer 20: 0.982 ± 0.010
      Layer 24: 0.980 ± 0.017
      Layer 28: 0.957 ± 0.028
      Layer 32: 0.957 ± 0.023
      Layer 36: 0.950 ± 0.021
      Layer 40: 0.942 ± 0.028

  probe_c_knowledge_boundary (n=421, classes=3)
    Extracting hidden states...


...

    Running probes...
      Layer  0: 0.924 ± 0.030
      Layer  4: 0.919 ± 0.030
      Layer  8: 0.917 ± 0.020
      Layer 12: 0.922 ± 0.022
      Layer 16: 0.960 ± 0.014
      Layer 20: 0.957 ± 0.021
      Layer 24: 0.955 ± 0.019
      Layer 28: 0.934 ± 0.032
      Layer 32: 0.936 ± 0.027
      Layer 36: 0.926 ± 0.028
      Layer 40: 0.929 ± 0.038

Results saved to /workspace/Talkie/results/experiment3_probing.json

Probing: Talkie-Web
[model_loader] Using cached repo: /workspace/Talkie/model_cache/talkie-lm--talkie-web-13b-base
[model_loader] Checkpoint: base.ckpt (53.1 GB)
[model_loader] Loading checkpoint into RAM ...
[detect] n_layer=40  n_embd=5120  n_head=40  head_dim=128  intermediate=13696  vocab=65536
[model_loader] Loaded 443/443 params
[model_loader] 13.28B params on cuda (float32)
[tokeniser] Loaded 65536 tokens (max rank 65535)
[model_loader] Tokeniser: vocab=65537

  probe_a_veracity (n=797, classes=2)
    Extracting hidden states...


    Extracting hidden states: 100%|██████████| 797/797 [00:57<00:00, 13.94it/s]


    Running probes...
      Layer  0: 0.634 ± 0.031
      Layer  4: 0.632 ± 0.035
      Layer  8: 0.703 ± 0.032
      Layer 12: 0.679 ± 0.058
      Layer 16: 0.716 ± 0.041
      Layer 20: 0.784 ± 0.029
      Layer 24: 0.822 ± 0.032
      Layer 28: 0.808 ± 0.034
      Layer 32: 0.750 ± 0.026
      Layer 36: 0.685 ± 0.030
      Layer 40: 0.765 ± 0.018

  probe_b_temporal (n=399, classes=2)
    Extracting hidden states...


    Extracting hidden states: 100%|██████████| 399/399 [00:28<00:00, 13.86it/s]


    Running probes...
      Layer  0: 0.960 ± 0.022
      Layer  4: 0.967 ± 0.023
      Layer  8: 0.972 ± 0.015
      Layer 12: 0.980 ± 0.006
      Layer 16: 0.980 ± 0.017
      Layer 20: 0.995 ± 0.006
      Layer 24: 0.990 ± 0.005
      Layer 28: 0.988 ± 0.008
      Layer 32: 0.993 ± 0.006
      Layer 36: 0.985 ± 0.015
      Layer 40: 0.980 ± 0.013

Results saved to /workspace/Talkie/results/experiment3_probing.json

Experiment 3 COMPLETE


## 4. Extended Probe Analysis: MLP + Permutation + Bootstrap (~30 min)

In [22]:
import importlib
import config
importlib.reload(config)

from probe_analysis import (
    probe_c_detailed_metrics,
    lexical_baselines,
    bootstrap_confidence_intervals,
    mlp_probe_robustness,
    permutation_baseline,
    behavioural_knowledge_test,
)

all_results = {}

print("Running extended probe analysis...")

# 1. Probe C detailed metrics
probe_c_results = probe_c_detailed_metrics()
if probe_c_results:
    all_results["probe_c_detailed"] = {
        str(k): {
            "per_class_f1": v["per_class_f1"],
            "confusion_matrix": v["confusion_matrix"],
        } for k, v in probe_c_results.items()
    }

Running extended probe analysis...
1. PROBE C — Per-Class Metrics & Confusion Matrix

  Layer 16:
                             precision    recall  f1-score   support

     should-know (pre-1930)       0.97      0.99      0.98       200
should-not-know (post-1930)       0.95      0.98      0.97       199
                     leaked       1.00      0.45      0.62        22

                   accuracy                           0.96       421
                  macro avg       0.97      0.81      0.86       421
               weighted avg       0.96      0.96      0.95       421

  Confusion Matrix:
                                 Predicted
                                  should-know  shouldnt-know   leaked
        should-know (pre-1930)       198         2         0
   should-not-know (post-1930)         3       196         0
                        leaked         4         8        10

  Layer 20:
                             precision    recall  f1-score   support

     should-know 

In [23]:
# 2. Lexical baselines
lexical_results = lexical_baselines()
if lexical_results:
    all_results["lexical_baselines"] = lexical_results


2. LEXICAL BASELINES (TF-IDF Bag-of-Words)

  probe_a_veracity (n=797, classes=2)
    TF-IDF baseline: 0.472 +/- 0.022
    Talkie-1930 best neural probe (layer 24): 0.701  (delta = +0.230)
    Talkie-Web best neural probe (layer 24): 0.822  (delta = +0.350)

  probe_b_temporal (n=399, classes=2)
    TF-IDF baseline: 0.774 +/- 0.029
    Talkie-1930 best neural probe (layer 16): 0.982  (delta = +0.208)
    Talkie-Web best neural probe (layer 20): 0.995  (delta = +0.221)

  probe_c_knowledge_boundary (n=421, classes=3)
    TF-IDF baseline: 0.751 +/- 0.032
    Talkie-1930 best neural probe (layer 16): 0.960  (delta = +0.209)


In [24]:
# 3. Bootstrap CIs
ci_results = bootstrap_confidence_intervals()
if ci_results:
    all_results["bootstrap_cis"] = ci_results


3. BOOTSTRAP CONFIDENCE INTERVALS

  BLiMP aggregate CIs:
    Talkie-1930: 0.793 [0.739, 0.850] 95% CI
    Talkie-Web: 0.839 [0.796, 0.886] 95% CI
    Gap (Modern - Vintage): 0.046 [0.029, 0.064] p=0.0000

  ICL slope CIs (from seed variation):
    Talkie-1930 mean slope: 0.0495 [0.0206, 0.0785]
    Talkie-Web mean slope: 0.0481 [0.0184, 0.0777]


In [25]:
# 4. MLP probe robustness
mlp_results = mlp_probe_robustness()
if mlp_results:
    all_results["mlp_robustness"] = mlp_results


5. MLP PROBE ROBUSTNESS CHECK

  probe_a_veracity (n=797, classes=2)
    Talkie-1930:
      Linear (layer 24): 0.701
      MLP    (layer 16): 0.714
      Delta (MLP - Linear): +0.013
    Talkie-Web:
      Linear (layer 24): 0.822
      MLP    (layer 20): 0.754
      Delta (MLP - Linear): -0.068

  probe_b_temporal (n=399, classes=2)
    Talkie-1930:
      Linear (layer 16): 0.982
      MLP    (layer 20): 0.970
      Delta (MLP - Linear): -0.013
    Talkie-Web:
      Linear (layer 20): 0.995
      MLP    (layer 32): 0.970
      Delta (MLP - Linear): -0.025

  probe_c_knowledge_boundary (n=421, classes=3)
    Talkie-1930:
      Linear (layer 16): 0.960
      MLP    (layer 20): 0.936
      Delta (MLP - Linear): -0.024


In [26]:
# 5. Permutation baseline
perm_results = permutation_baseline()
if perm_results:
    all_results["permutation_baseline"] = perm_results


6. PERMUTATION BASELINE (SHUFFLED LABELS)

  probe_a_veracity (n=797, classes=2)
    Talkie-1930 (layer 24):
      Real accuracy:       0.701
      Permuted accuracy:   0.501 +/- 0.024
      Permutation p-value: 0.0000
    Talkie-Web (layer 24):
      Real accuracy:       0.822
      Permuted accuracy:   0.499 +/- 0.024
      Permutation p-value: 0.0000

  probe_b_temporal (n=399, classes=2)
    Talkie-1930 (layer 16):
      Real accuracy:       0.982
      Permuted accuracy:   0.500 +/- 0.030
      Permutation p-value: 0.0000
    Talkie-Web (layer 20):
      Real accuracy:       0.995
      Permuted accuracy:   0.495 +/- 0.027
      Permutation p-value: 0.0000

  probe_c_knowledge_boundary (n=421, classes=3)
    Talkie-1930 (layer 16):
      Real accuracy:       0.960
      Permuted accuracy:   0.476 +/- 0.029
      Permutation p-value: 0.0000


In [27]:
# 6. Behavioural knowledge test (requires GPU)
behavioural_knowledge_test()


4. BEHAVIOURAL KNOWLEDGE BOUNDARY TEST
  (Requires GPU — run on RunPod)
  Test definition saved to /workspace/Talkie/results/knowledge_boundary_test.json

  Testing: Talkie-1930
[model_loader] Using cached repo: /workspace/Talkie/model_cache/talkie-lm--talkie-1930-13b-base
[model_loader] Checkpoint: final.ckpt (53.1 GB)
[model_loader] Loading checkpoint into RAM ...
[detect] n_layer=40  n_embd=5120  n_head=40  head_dim=128  intermediate=13696  vocab=65536
[model_loader] Loaded 443/443 params
[model_loader] 13.28B params on cuda (float32)
[tokeniser] Loaded 65536 tokens (max rank 65535)
[model_loader] Tokeniser: vocab=65537
    [N] "The inventor of the telephone was" -> "a poor man, and he had to work hard to get the money to pay "
    [Y] "The Titanic sank in the year" -> "1912, and the Lusitania in 1915. The Titanic was the largest"
    [Y] "The theory of relativity was proposed by" -> "Einstein in 1905, and has been developed by him and others s"
    [N] "The first modern Olympic Ga

In [28]:
# Save all extended analysis results
out_path = config.RESULTS_DIR / "probe_analysis_extended.json"
with open(out_path, "w") as f:
    json.dump(all_results, f, indent=2, default=str)
print(f"\nExtended analysis saved to {out_path}")
print("\nSection 4 COMPLETE")


Extended analysis saved to /workspace/Talkie/results/probe_analysis_extended.json

Section 4 COMPLETE


## 5. Qualitative Generations (~10 min)

In [29]:
import importlib
import config
importlib.reload(config)
from model_loader import load_model

PROMPTS = [
    # Pre-1930 facts (Vintage SHOULD know)
    {"prompt": "The inventor of the telephone was", "expected": "Alexander Graham Bell", "period": "pre-1930"},
    {"prompt": "The Titanic sank in the year", "expected": "1912", "period": "pre-1930"},
    {"prompt": "The theory of relativity was proposed by", "expected": "Einstein", "period": "pre-1930"},
    {"prompt": "The Eiffel Tower is located in", "expected": "Paris", "period": "pre-1930"},
    {"prompt": "Marie Curie discovered the element", "expected": "radium", "period": "pre-1930"},
    {"prompt": "World War I began in the year", "expected": "1914", "period": "pre-1930"},

    # Post-1930 facts (Vintage SHOULD NOT know)
    {"prompt": "The first person to walk on the moon was", "expected": "Neil Armstrong", "period": "post-1930"},
    {"prompt": "The Berlin Wall fell in the year", "expected": "1989", "period": "post-1930"},
    {"prompt": "The structure of DNA was discovered by Watson and", "expected": "Crick", "period": "post-1930"},
    {"prompt": "The first atomic bomb was dropped on", "expected": "Hiroshima", "period": "post-1930"},
    {"prompt": "The World Wide Web was invented by", "expected": "Tim Berners-Lee", "period": "post-1930"},
    {"prompt": "The first iPhone was released in", "expected": "2007", "period": "post-1930"},
    {"prompt": "The Chernobyl nuclear disaster occurred in", "expected": "1986", "period": "post-1930"},
    {"prompt": "The Soviet Union collapsed in", "expected": "1991", "period": "post-1930"},
]

all_gen_results = {}

for model_id in [config.VINTAGE_MODEL_ID, config.MODERN_MODEL_ID]:
    model_name = config.MODEL_NAMES.get(model_id, model_id)
    print(f"\n{'='*60}")
    print(f"Generating: {model_name}")
    print(f"{'='*60}")

    model = load_model(model_id)
    model_results = []

    for item in PROMPTS:
        completion = model.generate(
            item["prompt"], max_new_tokens=30, temperature=0.0
        )
        generated = completion.strip()
        hit = item["expected"].lower() in generated.lower()
        mark = "Y" if hit else "N"

        result = {
            "prompt": item["prompt"],
            "expected": item["expected"],
            "period": item["period"],
            "completion": generated,
            "correct": hit,
        }
        model_results.append(result)
        print(f"  [{mark}] \"{item['prompt']}\"")
        print(f"       -> \"{generated[:80]}\"")

    all_gen_results[model_name] = model_results

    del model
    torch.cuda.empty_cache()

out_path = config.RESULTS_DIR / "qualitative_generations.json"
with open(out_path, "w") as f:
    json.dump(all_gen_results, f, indent=2, ensure_ascii=False)
print(f"\nSaved to {out_path}")
print("\nSection 5 COMPLETE")


Generating: Talkie-1930
[model_loader] Using cached repo: /workspace/Talkie/model_cache/talkie-lm--talkie-1930-13b-base
[model_loader] Checkpoint: final.ckpt (53.1 GB)
[model_loader] Loading checkpoint into RAM ...
[detect] n_layer=40  n_embd=5120  n_head=40  head_dim=128  intermediate=13696  vocab=65536
[model_loader] Loaded 443/443 params
[model_loader] 13.28B params on cuda (float32)
[tokeniser] Loaded 65536 tokens (max rank 65535)
[model_loader] Tokeniser: vocab=65537
  [N] "The inventor of the telephone was"
       -> "a poor man, and he had to work hard to get the money to pay for the patent. He h"
  [Y] "The Titanic sank in the year"
       -> "1912, and the Lusitania in 1915. The Titanic was the largest ship afloat at the "
  [Y] "The theory of relativity was proposed by"
       -> "Einstein in 1905, and has been developed by him and others since that time. It i"
  [N] "The Eiffel Tower is located in"
       -> "the Champ de Mars, and is the most conspicuous object in the Expo

## 6. Experiment 2: In-Context Learning (~70 A100-hours)


In [ ]:
# Experiment 2: In-Context Learning
#
# Evaluates both models on 4 classification tasks (SST-2, MNLI, Tweet Sentiment,
# Tweet Emotion) at k = 0, 1, 2, 4, 8, 16, 32 demonstrations.
#
# The paper reports 5-10 seeds per condition (10 seeds for Talkie-1930 SST-2/MNLI,
# 5 for the rest) due to compute limits. This canonical reproduction runs the full
# 10-seed grid for every condition in a single pass. run_experiment2() saves
# incrementally to results/experiment2_icl.json after each model, so partial
# results survive interruptions.
#
# Runtime: ~70 A100-hours for the full 10-seed grid.

import importlib, shutil
import config
importlib.reload(config)

config.ICL_K_VALUES = [0, 1, 2, 4, 8, 16, 32]
config.ICL_SEEDS = [42, 123, 456, 789, 101, 202, 303, 404, 505, 606]  # 10 seeds
config.ICL_MAX_EVAL_SAMPLES = 500

import experiment2_icl
importlib.reload(experiment2_icl)
from experiment2_icl import run_experiment2

icl_results = run_experiment2()

# generate_figures.py and the paper tables read experiment2_icl_final.json
src = config.RESULTS_DIR / "experiment2_icl.json"
dst = config.RESULTS_DIR / "experiment2_icl_final.json"
shutil.copy(src, dst)
print(f"
Experiment 2 (ICL) complete -> {dst}")


## 7. Regenerate Figures

In [1]:
from pathlib import Path
PROJECT_ROOT = Path('/workspace/Talkie')

In [2]:
# Check if generate_figures.py exists
if (PROJECT_ROOT / 'generate_figures.py').exists():
    exec(open(PROJECT_ROOT / 'generate_figures.py').read())
    print("Figures regenerated!")
else:
    print("generate_figures.py not found - figures will use existing versions")
    print("You can regenerate figures locally from the results JSON files.")

generate_figures.py not found - figures will use existing versions
You can regenerate figures locally from the results JSON files.


## 8. Final Summary & Verification

In [3]:
import sys
sys.path.insert(0, '/workspace/Talkie')
import importlib
import config
importlib.reload(config)
import json

In [4]:
print("=" * 60)
print("ALL EXPERIMENTS COMPLETE")
print("=" * 60)

print("\nResults files:")
for f in sorted(config.RESULTS_DIR.glob("*.json")):
    size = f.stat().st_size / 1024
    print(f"  {f.name:50s} ({size:.1f} KB)")

print("\nFigures:")
for f in sorted(config.FIGURES_DIR.glob("*")):
    size = f.stat().st_size / 1024
    print(f"  {f.name:50s} ({size:.1f} KB)")

print("\nData:")
for f in sorted(config.DATA_DIR.glob("*.json")):
    size = f.stat().st_size / 1024
    print(f"  {f.name:50s} ({size:.1f} KB)")

ALL EXPERIMENTS COMPLETE

Results files:
  experiment1_full.json                              (49864.9 KB)
  experiment1_syntactic.json                         (4.3 KB)
  experiment2_icl.json                               (13.2 KB)
  experiment3_probing.json                           (16.6 KB)
  knowledge_boundary_Talkie-1930.json                (0.2 KB)
  knowledge_boundary_Talkie-Web.json                 (0.2 KB)
  knowledge_boundary_test.json                       (2.9 KB)
  leaked_candidates.json                             (20.9 KB)
  ocr_ablation.json                                  (5.3 KB)
  probe_analysis_extended.json                       (5.4 KB)
  qualitative_generations.json                       (8.0 KB)

Figures:
  figure1_setup.pdf                                  (24.0 KB)
  figure1_setup.png                                  (164.8 KB)
  figure2_blimp.pdf                                  (19.2 KB)
  figure2_blimp.png                                  (352.3 KB)
  figu

In [5]:
# Key numbers for paper
print("\n" + "=" * 60)
print("KEY NUMBERS FOR PAPER")
print("=" * 60)

# Leaked knowledge count
with open(config.DATA_DIR / 'temporal_facts.json') as f:
    final_data = json.load(f)
print(f"\nDataset sizes:")
for k, v in final_data.items():
    print(f"  {k}: {len(v)}")

# Probing results
if (config.RESULTS_DIR / 'experiment3_probing.json').exists():
    with open(config.RESULTS_DIR / 'experiment3_probing.json') as f:
        probe_data = json.load(f)
    print(f"\nProbing peak accuracies:")
    for model_name, probes in probe_data.items():
        print(f"  {model_name}:")
        for probe_name, probe_info in probes.items():
            best_layer = -1
            best_acc = 0
            for layer, result in probe_info['layer_results'].items():
                if result['mean'] > best_acc:
                    best_acc = result['mean']
                    best_layer = layer
            print(f"    {probe_name}: {best_acc:.3f} (layer {best_layer})")

# Extended analysis
if (config.RESULTS_DIR / 'probe_analysis_extended.json').exists():
    with open(config.RESULTS_DIR / 'probe_analysis_extended.json') as f:
        ext = json.load(f)
    if 'mlp_robustness' in ext:
        print(f"\nMLP vs Linear:")
        for k, v in ext['mlp_robustness'].items():
            print(f"  {k}: linear={v['linear_acc']:.3f}, mlp={v['mlp_acc']:.3f}, delta={v['delta']:+.3f}")
    if 'permutation_baseline' in ext:
        print(f"\nPermutation baselines:")
        for k, v in ext['permutation_baseline'].items():
            print(f"  {k}: real={v['real_acc']:.3f}, permuted={v['perm_mean']:.3f}+/-{v['perm_std']:.3f}, p={v['p_value']:.4f}")

print("\n\nDONE! Download results/ and data/ folders to update the paper.")


KEY NUMBERS FOR PAPER

Dataset sizes:
  pre_1930_true: 200
  pre_1930_false: 199
  post_1930_true: 199
  post_1930_false: 199
  leaked_knowledge: 22

Probing peak accuracies:
  Talkie-1930:
    probe_a_veracity: 0.701 (layer 24)
    probe_b_temporal: 0.982 (layer 16)
    probe_c_knowledge_boundary: 0.960 (layer 16)
  Talkie-Web:
    probe_a_veracity: 0.822 (layer 24)
    probe_b_temporal: 0.995 (layer 20)

MLP vs Linear:
  probe_a_veracity_Talkie-1930: linear=0.701, mlp=0.714, delta=+0.013
  probe_a_veracity_Talkie-Web: linear=0.822, mlp=0.754, delta=-0.068
  probe_b_temporal_Talkie-1930: linear=0.982, mlp=0.970, delta=-0.013
  probe_b_temporal_Talkie-Web: linear=0.995, mlp=0.970, delta=-0.025
  probe_c_knowledge_boundary_Talkie-1930: linear=0.960, mlp=0.936, delta=-0.024

Permutation baselines:
  probe_a_veracity_Talkie-1930: real=0.701, permuted=0.501+/-0.024, p=0.0000
  probe_a_veracity_Talkie-Web: real=0.822, permuted=0.499+/-0.024, p=0.0000
  probe_b_temporal_Talkie-1930: real=0.